# IPL Auction Price Prediction

**Goal:** Predict a player's IPL auction price. Starting with a Linear Regression baseline, then implement and evaluate additional algorithms one by one, comparing all of them on identical metrics to pick the best model.

**Primary evaluation metric:** Mean Absolute Error (MAE) — interpretable in the same unit as price, and less skewed by extreme high-value player outliers than RMSE. R² and RMSE are also tracked for completeness.

---

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

#Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

## 2. Load Data

In [2]:
df=pd.read_csv("Modified_dataset.csv")

In [3]:
df.head()

,Unnamed: 0,Name,Age,Country,Role,Matches,Experience ( in Yrs),Runs,Strike_Rate,Wickets,Economy,Average,Team_Preference,Auction_Price_Simulated ( in Cr )
0,0,MS Dhoni,42,India,Wicketkeeper,350,20,12000,135.0,0,0.00,50.00,CSK,19.42
1,1,Sachin Tendulkar,45,India,Batter,400,24,18000,125.0,5,8.50,45.00,MI,20.00
2,2,Virat Kohli,36,India,Batter,300,16,13000,138.0,2,7.00,52.00,RCB,20.00
3,3,Rohit Sharma,37,India,Batter,280,17,14000,132.0,0,8.00,48.00,MI,20.00
4,4,Jasprit Bumrah,35,India,Bowler,196,7,2661,144.9,798,7.69,27.43,MI,16.09


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Unnamed: 0                         12000 non-null  int64  
 1   Name                               12000 non-null  str    
 2   Age                                12000 non-null  int64  
 3   Country                            12000 non-null  str    
 4   Role                               12000 non-null  str    
 5   Matches                            12000 non-null  int64  
 6   Experience ( in Yrs)               12000 non-null  int64  
 7   Runs                               12000 non-null  int64  
 8   Strike_Rate                        12000 non-null  float64
 9   Wickets                            12000 non-null  int64  
 10  Economy                            12000 non-null  float64
 11  Average                            12000 non-null  float64
 12  T

In [5]:
df.describe()

,Unnamed: 0,Age,Matches,Experience ( in Yrs),Runs,Strike_Rate,Wickets,Economy,Average,Auction_Price_Simulated ( in Cr )
count,12000.00000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000,12000.000000
mean,5999.50000,30.094667,146.447167,6.523083,5897.813833,125.118867,191.803500,5.068873,37.386248,11.888975
std,3464.24595,7.191883,125.774590,5.361237,3630.126361,17.270469,246.319993,3.167495,10.094644,3.118079
min,0.00000,18.000000,15.000000,1.000000,0.000000,95.000000,0.000000,0.000000,20.000000,2.190000
25%,2999.75000,24.000000,46.000000,2.000000,2986.750000,110.300000,0.000000,0.000000,28.570000,9.640000
50%,5999.50000,30.000000,108.000000,5.000000,5359.500000,125.200000,15.000000,6.220000,37.340000,11.760000
75%,8999.25000,36.000000,217.000000,10.000000,8450.250000,140.100000,351.000000,7.420000,46.092500,13.980000
max,11999.00000,45.000000,713.000000,24.000000,18000.000000,155.000000,900.000000,9.500000,55.000000,20.000000


In [6]:
df=df.drop(columns=['Unnamed: 0'])

In [7]:
df.isnull().sum()

Name                                 0
Age                                  0
Country                              0
Role                                 0
Matches                              0
Experience ( in Yrs)                 0
Runs                                 0
Strike_Rate                          0
Wickets                              0
Economy                              0
Average                              0
Team_Preference                      0
Auction_Price_Simulated ( in Cr )    0
dtype: int64

## 3.Define Input features and Target variable

In [8]:
X = df.drop(columns=["Auction_Price_Simulated ( in Cr )"])  # input features
y = df["Auction_Price_Simulated ( in Cr )"]                 # target varible

## 4.Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)